In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import pandas as pd
import os
import pickle
import config

from tqdm.auto import tqdm
from spacy.matcher import PhraseMatcher
from spacy.util import filter_spans


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report

from sklearn.decomposition import TruncatedSVD

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from xgboost import XGBRanker

from sklearn.model_selection import GroupKFold

In [4]:
from src.features import *
from src.metric import model_evaluation
from src.tech_aliases import tech_aliases


In [5]:
with open(config.ALL_SKILLS_PKL, 'rb') as f:
    skills = pickle.load(f)

In [6]:
with open(os.path.join(config.CLEANED_DATA_DIR,'train_df.pkl'),'rb') as f:
    train_df=pickle.load(f)
    
with open(os.path.join(config.CLEANED_DATA_DIR,'val_df.pkl'),'rb') as f:
    val_df=pickle.load(f)
        
with open(os.path.join(config.CLEANED_DATA_DIR,'test_df.pkl'),'rb') as f:
    test_df=pickle.load(f)
    


In [7]:
np.random.seed(config.SEED)

In [8]:
# train_df['resume_clean']=train_df['resume_text'].apply(lambda x: further_clean_text(x))
# val_df['resume_clean']=val_df['resume_text'].apply(lambda x: further_clean_text(x))
# test_df['resume_clean']=test_df['resume_text'].apply(lambda x: further_clean_text(x))

# train_df['jd_clean']=train_df['job_description_text'].apply(lambda x: further_clean_text(x))
# val_df['jd_clean']=val_df['job_description_text'].apply(lambda x: further_clean_text(x))
# test_df['jd_clean']=test_df['job_description_text'].apply(lambda x: further_clean_text(x))

In [9]:
train_df.head(2)

,resume_text,job_description_text,label,resume_exp,jd_exp,resume_clean,jd_clean
0,SummaryHighly motivated Sales Associate with e...,Net2Source Inc. is an award-winning total work...,0,0.0,9.0,summaryhighly motivated sales associate with e...,net2source inc. is an award-winning total work...
1,Professional SummaryCurrently working with Cat...,At Salas OBrien we tell our clients that were ...,0,0.0,3.0,professional summarycurrently working with cat...,at salas obrien we tell our clients that were ...


In [10]:
import spacy

try:
    spacy.load("en_core_web_md")
except OSError:
    print("Downloading spaCy model 'en_core_web_md'...")
    !python -m spacy download en_core_web_md
    import spacy

In [11]:
def build_phrase_matcher(nlp, alias_dict,skills):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    patterns=[]
    skill_map={}
    # Add alias dict
    for key, values in alias_dict.items():
        patterns.append(nlp.make_doc(key))
        skill_map[key.lower()]=key
        for v in values:
            patterns.append(nlp.make_doc(v))
            skill_map[v.lower()]=key

     # 2. Add remaining skills (from Kaggle)
    for skill in skills:
        if skill.lower() not in skill_map:
            patterns.append(nlp.make_doc(skill))
            skill_map[skill.lower()] = skill


    matcher.add("SKILLS", patterns)
    return matcher,skill_map


In [12]:
def single_pass_pipeline(texts, matcher,skill_map, nlp):
    processed_texts=[]
    print("Applying alias normalization and phrase matching...")


    for doc in tqdm(nlp.pipe(texts, batch_size=200,n_process=-1), total=len(texts)):
        raw_matches=matcher(doc)
        spans=[(match_id, start, end) for match_id, start, end in raw_matches]
        filtered_spans=filter_spans([doc[start:end] for _, start, end in spans])

        span_dict={}
        match_map={(start, end): match_id for match_id, start, end in spans}

        for span in filtered_spans:
            match_id=match_map[(span.start, span.end)]
            normalize_skill=skill_map.get(span.text.lower(),span.text.lower())
            span_dict[span.start]=(span.end, normalize_skill)

        tokens=[]
        i=0

        while i < len(doc):
            if i in span_dict:
                end_idx, label=span_dict[i]
                tokens.append(label.replace(" ","_"))
                i=end_idx
                continue

            token=doc[i]

            if (token.is_stop or token.is_punct or token.is_space or
                token.like_url or token.like_email):
                i += 1
                continue

            if token.pos_ not in ['NOUN', 'VERB', 'ADJ', 'PROPN']:
                i += 1
                continue

            lemma=token.lemma_.lower()

            if len(lemma) > 2 :
                tokens.append(lemma)

            i += 1

        processed_texts.append(" ".join(tokens))

    return processed_texts


In [13]:
def preprocess_pipeline(texts, alias_dict,skills):
    nlp = spacy.load("en_core_web_md",disable=["parser", "ner"])
    print("Building phrase matcher...")
    matcher,skill_map =build_phrase_matcher(nlp, alias_dict,skills)
    return single_pass_pipeline(texts,matcher,skill_map,nlp)

In [ ]:
resume_processed_train = preprocess_pipeline(train_df['resume_text'].tolist(), tech_aliases,skills)
jd_processed_train = preprocess_pipeline(train_df['job_description_text'].tolist(), tech_aliases,skills)

jd_processed_val = preprocess_pipeline(val_df['job_description_text'].tolist(), tech_aliases,skills)
resume_processed_val = preprocess_pipeline(val_df['resume_text'].tolist(), tech_aliases,skills)


resume_processed_test = preprocess_pipeline(test_df['resume_text'].tolist(), tech_aliases,skills)
jd_processed_test = preprocess_pipeline(test_df['job_description_text'].tolist(), tech_aliases,skills)


Building phrase matcher...
Applying alias normalization and phrase matching...


  0%|          | 0/6240 [00:00<?, ?it/s]

Building phrase matcher...
Applying alias normalization and phrase matching...


  0%|          | 0/6240 [00:00<?, ?it/s]

Building phrase matcher...
Applying alias normalization and phrase matching...


...

  0%|          | 0/1167 [00:00<?, ?it/s]

Building phrase matcher...
Applying alias normalization and phrase matching...


  0%|          | 0/1759 [00:00<?, ?it/s]

Building phrase matcher...
Applying alias normalization and phrase matching...


  0%|          | 0/1759 [00:00<?, ?it/s]

In [15]:
preprocessed_directory_path=config.PREPROCESSED_DATA_DIR

In [16]:
# resume_processed_train = pickle.load(open(os.path.join(preprocessed_directory_path,'train_resume.pkl'),'rb'))
# jd_processed_train = pickle.load(open(os.path.join(preprocessed_directory_path,'train_jd.pkl'),'rb'))

# jd_processed_val = pickle.load(open(os.path.join(preprocessed_directory_path,'val_jd.pkl'),'rb'))
# resume_processed_val = pickle.load(open(os.path.join(preprocessed_directory_path,'val_resume.pkl'),'rb'))

# resume_processed_test = pickle.load(open(os.path.join(preprocessed_directory_path,'test_resume.pkl'),'rb'))
# jd_processed_test = pickle.load(open(os.path.join(preprocessed_directory_path,'test_jd.pkl'),'rb'))


In [20]:
os.makedirs(preprocessed_directory_path,exist_ok=True)

pickle.dump(resume_processed_train,open(preprocessed_directory_path + '/train_resume.pkl','wb'))
pickle.dump(resume_processed_val,open(preprocessed_directory_path + '/val_resume.pkl','wb'))
pickle.dump(resume_processed_test,open(preprocessed_directory_path + '/test_resume.pkl','wb'))

pickle.dump(jd_processed_train, open(preprocessed_directory_path + '/train_jd.pkl','wb'))
pickle.dump(jd_processed_val, open(preprocessed_directory_path + '/val_jd.pkl','wb'))
pickle.dump(jd_processed_test, open(preprocessed_directory_path + '/test_jd.pkl','wb'))

In [21]:
train_df['resume_clean']=resume_processed_train
train_df['jd_clean']=jd_processed_train

val_df['resume_clean']=resume_processed_val
val_df['jd_clean']=jd_processed_val

test_df['resume_clean']=resume_processed_test
test_df['jd_clean']=jd_processed_test

In [22]:
train_df.head()

,resume_text,job_description_text,label,resume_exp,jd_exp,resume_clean,jd_clean
0,SummaryHighly motivated Sales Associate with e...,Net2Source Inc. is an award-winning total work...,0,0.0,9.0,summaryhighly motivate sales associate extensi...,net2source inc. award win total workforce solu...
1,Professional SummaryCurrently working with Cat...,At Salas OBrien we tell our clients that were ...,0,0.0,3.0,professional summarycurrently work caterpillar...,salas obrien tell clients engineer impact pass...
2,SummaryI started my construction career in Jun...,Schweitzer Engineering Laboratories (SEL) Infr...,0,0.0,6.0,summaryi start construction career june jackso...,schweitzer engineering laboratories sel infras...
3,SummaryCertified Electrical Foremanwith thirte...,"Mizick Miller & Company, Inc. is looking for a...",0,8.0,0.0,summarycertified electrical foremanwith year e...,mizick miller company inc. look dynamic indivi...
4,SummaryWith extensive experience in business/r...,Life at Capgemini\nCapgemini supports all aspe...,0,5.0,5.0,summarywith extensive experience business requ...,life capgemini capgemini support aspect change...


In [23]:
train_df=train_df.copy()

In [24]:
train_df = train_df.sort_values('job_description_text').reset_index(drop=True)
val_df = val_df.sort_values('job_description_text').reset_index(drop=True)

In [25]:
y_train=train_df['label']
y_val=val_df['label']

In [26]:
vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    stop_words='english'
)

vectorizer.fit(pd.concat([train_df['resume_clean'],train_df['jd_clean']]))
X_train_resume = vectorizer.transform(train_df['resume_clean'])
X_train_jd = vectorizer.transform(train_df['jd_clean'])

X_val_resume = vectorizer.transform(val_df['resume_clean'])
X_val_jd = vectorizer.transform(val_df['jd_clean'])


In [27]:
train_sim = cosine_similarity(X_train_resume, X_train_jd).diagonal().reshape(-1,1)
val_sim = cosine_similarity(X_val_resume, X_val_jd).diagonal().reshape(-1,1)


In [28]:
skill_overlap_train=skill_overlap(train_df['resume_clean'],train_df['jd_clean'],skills)
skill_overlap_val=skill_overlap(val_df['resume_clean'],val_df['jd_clean'],skills)


In [29]:
exp_gap_train=train_df['resume_exp']-train_df['jd_exp']
exp_gap_val=val_df['resume_exp']-val_df['jd_exp']

exp_match_train=train_df['resume_exp']/(train_df['jd_exp']+1)
exp_match_val=val_df['resume_exp']/(val_df['jd_exp']+1)

exp_enough_train=train_df['resume_exp']>=train_df['jd_exp'].astype(int)
exp_enough_val=val_df['resume_exp']>=val_df['jd_exp'].astype(int)


In [31]:
X_train_features = np.hstack([
    train_sim,
    skill_overlap_train,
    exp_gap_train.to_numpy().reshape(-1,1),
])

X_val_features = np.hstack([
    val_sim,
    skill_overlap_val,
    exp_gap_val.to_numpy().reshape(-1,1),
])

In [32]:
pd.DataFrame(X_train_features).corr()

,0,1,2,3
0,1.000000,0.476704,0.424530,-0.064358
1,0.476704,1.000000,0.365110,-0.102671
2,0.424530,0.365110,1.000000,0.013447
3,-0.064358,-0.102671,0.013447,1.000000


In [33]:
X_train_interaction = X_train_resume.multiply(X_train_jd)
X_val_interaction = X_val_resume.multiply(X_val_jd)


In [34]:
svd = TruncatedSVD(n_components=30, random_state=42)
X_train_svd = svd.fit_transform(X_train_interaction)
X_val_svd = svd.transform(X_val_interaction)

X_train_combine = np.hstack([X_train_svd, X_train_features])
X_val_combine = np.hstack([X_val_svd, X_val_features])

scaler = StandardScaler()
X_train_final = scaler.fit_transform(X_train_combine)
X_val_final = scaler.transform(X_val_combine)


In [35]:
models={"SVM":SVC(class_weight="balanced",probability=True),
        "LogisticRegression":LogisticRegression(class_weight="balanced",max_iter=1000),
        "RandomForest": RandomForestClassifier(n_estimators=200,class_weight="balanced", random_state=42),
        "XGB":XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1)}

In [36]:
def train_eval_classify(X_train, X_test, y_train, y_test, model, df):
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)

  print("Unique predictions:", set(y_pred))
  print("Variance:", np.var(y_pred))

  y_prob = model.predict_proba(X_test)
  scores=y_prob[:,2]

  print(classification_report(y_test, y_pred))

  print("Metrics:")
  model_evaluation(scores,df,'jd_clean')

In [37]:
for name,model_obj in models.items():
    print(f"\n--- Evaluating Model: {name} ","-"*50)
    train_eval_classify(X_train_final, X_val_final, y_train, y_val, model_obj, val_df)


--- Evaluating Model: SVM  --------------------------------------------------


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Unique predictions: {0, 1, 2}
Variance: 0.632453893085266
              precision    recall  f1-score   support

           0       0.67      0.82      0.74       613
           1       0.63      0.44      0.52       285
           2       0.48      0.41      0.44       269

    accuracy                           0.63      1167
   macro avg       0.60      0.55      0.57      1167
weighted avg       0.62      0.63      0.62      1167

Metrics:

--- Evaluating Model: LogisticRegression  --------------------------------------------------
Unique predictions: {0, 1, 2}
Variance: 0.6143907469698338
              precision    recall  f1-score   support

           0       0.65      0.77      0.71       613
           1       0.46      0.36      0.41       285
           2       0.38      0.30      0.33       269

    accuracy                           0.56      1167
   macro avg       0.50      0.48      0.48      1167
weighted avg       0.54      0.56      0.55      1167

Metrics:

--- Eval

In [38]:
xgb_rank=XGBRanker(objective='rank:ndcg',learning_rate=0.05,max_depth=7, n_estimators=300,
                  subsample=0.8,colsample_bytree=0.8)

group_train=train_df.groupby('jd_clean').size().to_list()
xgb_rank.fit(X_train_final,y_train,group=group_train)
scores=xgb_rank.predict(X_val_final)

In [39]:
print("Evaluating on Full Validation Df")
metrics=model_evaluation(scores,val_df,'jd_clean')
print(metrics)

Evaluating on Full Validation Df
{'spearman_score': 0.6558554131826023, 'topk_score': 1.0, 'ndcg_val': 0.849026213868921, 'mrr_score': 0.9756944444444443, 'map_score': 0.9289092151804148}


In [40]:
imp_feature=xgb_rank.feature_importances_
cosine_imp=imp_feature[0]
skill_overlap_imp=imp_feature[1:4]
exp_gap_imp=imp_feature[4]
svd_imp=imp_feature[5:].sum()

In [41]:
print("Cosine:", cosine_imp)
print("Skill_overlap:", skill_overlap_imp.sum())
print("Exp_gap:",exp_gap_imp)
print("SVD (semantic):", svd_imp)

Cosine: 0.058294844
Skill_overlap: 0.09819284
Exp_gap: 0.027249986
SVD (semantic): 0.81626236


In [42]:
kf=GroupKFold(n_splits=5)
ndcg_scores,spearman_scores,topk_scores,mrr_scores,map_scores=[],[],[],[],[]

for fold, (train_split, val_split) in enumerate(
    kf.split(X_train_final, y_train, groups=train_df['jd_clean'])):


    model=XGBRanker(objective='rank:ndcg',learning_rate=0.05,max_depth=7,
                       n_estimators=300,subsample=0.8,colsample_bytree=0.8)

    group_train=train_df.iloc[train_split].groupby('jd_clean').size().to_list()
    
    model.fit(X_train_final[train_split],y_train.iloc[train_split],group=group_train)
    xgb_scores=model.predict(X_train_final[val_split])

    print(f"fr{fold+1}:\n")
    metrics=model_evaluation(xgb_scores,train_df.iloc[val_split],"jd_clean")
    
    print("NDCG:",metrics['ndcg_val'])
    print("MAP:",metrics['map_score'])
    print("MRR:",metrics['mrr_score'])

    spearman_scores.append(metrics['spearman_score'])
    topk_scores.append(metrics['topk_score'])
    ndcg_scores.append(metrics['ndcg_val'])
    map_scores.append(metrics['map_score'])
    mrr_scores.append(metrics['mrr_score'])
    print("-"*100)

print(f"\nCV NDC:{np.mean(ndcg_scores):.4f}±{np.std(ndcg_scores):.4f}")
print(f"CV Spearma:{np.mean(spearman_scores):.4f}±{np.std(spearman_scores):.4f}")
print(f"CV Top-3 Accurac:{np.mean(topk_scores):.4f}±{np.std(topk_scores):.4f}")
print(f"CV MA:{np.mean(map_scores):.4f}±{np.std(map_scores):.4f}")
print(f"CV MR:{np.mean(mrr_scores):.4f}±{np.std(mrr_scores):.4f}")



fr1:

NDCG: 0.6482909023767001
MAP: 0.7140604846516274
MRR: 0.7875816993464053
----------------------------------------------------------------------------------------------------
fr2:

NDCG: 0.6477159215218283
MAP: 0.7160067568223772
MRR: 0.7812925170068028
----------------------------------------------------------------------------------------------------
fr3:

NDCG: 0.6285413723680809
MAP: 0.7269567246117377
MRR: 0.8216312056737589
----------------------------------------------------------------------------------------------------
fr4:

NDCG: 0.6419529128868057
MAP: 0.7353273699911385
MRR: 0.8162202380952381
----------------------------------------------------------------------------------------------------
fr5:

NDCG: 0.6662857825772386
MAP: 0.7768546632312222
MRR: 0.8248299319727891
----------------------------------------------------------------------------------------------------

CV NDC:0.6466±0.0122
CV Spearma:0.3350±0.0328
CV Top-3 Accurac:0.9306±0.0594
CV MA:0.7338±0.0228
CV

In [43]:
eval_df=val_df.copy()
eval_df['score']=scores

In [45]:
false_neg = eval_df[(eval_df['label'] == 2) & (eval_df['score'] < -0.3)]

print(f"Good Fit resumes scoring below -0.3: {len(false_neg)}")
print("\nSample false_neg resumes:")

i=0
for jd,group in false_neg.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:200]}")
    for _,row in group.head(3).iterrows():
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:300]}\n")
    i+=1
    if i==3:
        break
        

Good Fit resumes scoring below -0.3: 80

Sample false_neg resumes:
----------------------------------------------------------------------------------------------------
JD: alphabets moonshot factory diverse group inventor entrepreneur build launch technologies aim improve life million billion people goal 10x impact world intractable problem improvement approach project 
Score: -0.628
Resume: Summary•        
Over
Three years of extensive experience as a Front-End UI Developer with solid
understanding of database designing, development and installation of different
modules. 

•        
Professional
understanding of System development life cycle (SDLC) as well as various phases such as An

Score: -0.682
Resume: SummaryRecent graduate from Nucamp Coding Bootcamp with excellent research, technical and problem-solving skills. Detail-oriented and able to learn new technology quickly. Ambitious, career-focused job seeker, anxious to obtain entry-level Python Developer/DevOps position to help 

In [47]:
false_pos = eval_df[
    (eval_df['label'] == 0) & (eval_df['score'] > 1)]

print(f"Bad Fit resumes scoring above 1: {len(false_pos)}")
print("\nSample false_pos resumes:")

i=0
for jd,group in false_pos.groupby('jd_clean'):
    print("-"*100)
    print(f"JD: {jd[:300]}")
    for _,row in group.head(2).iterrows():
        print(f"Score: {row['score']:.3f}")
        print(f"Resume: {row['resume_text'][:500]}\n")
    i+=1
    if i==3:
        break
        

Bad Fit resumes scoring above 1: 2

Sample false_pos resumes:
----------------------------------------------------------------------------------------------------
JD: call innovator find future fiserv fiserv global leader fintech payments money information way moves world connect financial institution corporation merchant consumer million time day time swipe credit card pay mobile app withdraw money bank involve want impact global scale come difference fiserv suc
Score: 1.041
Resume: SUMMARYI am a full-stack developer with excellent communication and coordination skills and a passion for design.
EXPERIENCESoftware Engineer,07/2014-CurrentThe Nielsen Company–,,Founding member of the Innovation Lab. Led the team’s design and development of products for Unmanned Aerial Vehicles (UAVs) and virtual reality (VR).Skills: Java, python, caffe, CNN training, embedded software, MAVProxy, ArduPlane, project managementUAVs - Software Engineer & Operations SpecialistDeveloped sensing payl

---------

In [48]:
print("\nScore spread within groups:")
spreads = []
for jd, group in eval_df.groupby('jd_clean'):
    if len(group) > 1:
        spreads.append(group['score'].max() - group['score'].min())

print(f"Mean spread: {np.mean(spreads)}")
print(f"% groups with spread < 0.1:"f"{(np.array(spreads) < 0.1).mean()}")


Score spread within groups:
Mean spread: 3.4659903049468994
% groups with spread < 0.1:0.0


In [50]:
X_test_resume = vectorizer.transform(test_df['resume_clean'])
X_test_jd = vectorizer.transform(test_df['jd_clean'])

test_sim = cosine_similarity(X_test_resume, X_test_jd).diagonal().reshape(-1,1)

skill_overlap_test=skill_overlap(test_df['resume_clean'],test_df['jd_clean'],skills)

exp_gap_test=test_df['resume_exp']-test_df['jd_exp']

X_test_features = np.hstack([
    test_sim,
    skill_overlap_test,
    exp_gap_test.to_numpy().reshape(-1,1),
])

X_test_interaction = X_test_resume.multiply(X_test_jd)

X_test_svd = svd.transform(X_test_interaction)

X_test_combine = np.hstack([X_test_svd, X_test_features])

X_test_final = scaler.transform(X_test_combine)

final_scores=xgb_rank.predict(X_test_final)

metrics=model_evaluation(final_scores,test_df,'jd_clean')

In [51]:
print("Spearman:",metrics['spearman_score'])
print("Top-3 Accuracy:",metrics['topk_score'])
print("NDCG:",metrics['ndcg_val'])
print("MAP:",metrics['map_score'])
print("MRR:",metrics['mrr_score'])

Spearman: 0.2977361996477454
Top-3 Accuracy: 1.0
NDCG: 0.6368128687380836
MAP: 0.729572742250858
MRR: 0.8133879781420765


In [54]:
os.makedirs(os.path.join(config.MODELS_DIR,'XGB_RANK'),exist_ok=True)
xgb_rank.save_model(os.path.join(config.MODELS_DIR,'XGB_RANK','xgb_ranker.json'))

In [ ]:
#Finding->
# Data have noise , as tfidf try to lean common word so many resume jd pair have many similar word
# but not thieir semantic mean fully whole diffrent 
# also many jd resume may have noise as jd req electrical engineerinal and in resume chemical engirence
# and its related around work but still selected
# also many pair in boderline it may selected and may not selected it depend entirely on hr
#modle able to somewhat diffrentiate bw fit and no fi as within spread is high
